<a href="https://colab.research.google.com/github/JoGabTasca/ML/blob/main/Pipeline_Sinistros.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tarefa
Criar um pipeline para prever o tipo de sinistro usando modelos de IA. O pipeline deve incluir os seguintes passos: 1. Definição do Problema, 2. Coleta de Dados, 3. Divisão Inicial dos Dados, 4. Análise Exploratória (EDA), 5. Limpeza e Imputação dos Dados, 6. Engenharia de Atributos e 7. Balanceamento (se necessário). Será utilizado o dataset: "https://raw.githubusercontent.com/alvaroriz/datascience_datasets/refs/heads/main/NaoConformidadeTransportes.csv". Os dados serão divididos em 80% para treinamento e 20% para validação usando `train_test_split`. EDA em variáveis numéricas e verificar os tipos de dados. Tratar os valores nulos pesquisando e aplicar métodos apropriados com justificativa. Extrair atributos da coluna 'DATA' (por exemplo, período do dia, dia da semana), identificar feriados (biblioteca `holidays`) e excluir variáveis irrelevantes. Realizar EDA nos dados resultantes. Avaliar a necessidade de balanceamento e aplique se necessário.

## 1. Definição do Problema e 2. Coleta de Dados

### Definição do Problema

O objetivo deste projeto é desenvolver um pipeline de machine learning capaz de prever o tipo de sinistro com base em dados históricos de não conformidades em transportes. A categorização precisa dos sinistros é crucial para diversas frentes de uma operação de transporte, como a otimização de rotas, a alocação de recursos para prevenção, a gestão de riscos e a melhoria contínua dos processos. Ao prever o tipo de sinistro, as empresas podem agir proativamente para minimizar impactos negativos, reduzir custos e aumentar a eficiência.

A base de dados a ser utilizada, proveniente do link fornecido, contém informações sobre não conformidades em transportes e servirá como base para treinar e validar nossos modelos de IA. Através da análise e modelagem desses dados, buscaremos identificar padrões que permitam a previsão acurada do tipo de sinistro.

### Coleta de Dados

A coleta de dados para este projeto consiste em carregar o arquivo CSV disponível no link fornecido diretamente para um DataFrame da biblioteca pandas, que é amplamente utilizada para manipulação e análise de dados em Python.

In [ ]:
import pandas as pd

# URL da base de dados
url = "https://raw.githubusercontent.com/alvaroriz/datascience_datasets/refs/heads/main/NaoConformidadeTransportes.csv"

# Carregar os dados em um DataFrame, tentando diferentes delimitadores e ignorando linhas com erro
try:
    # Tentativa com tabulação como delimitador
    df = pd.read_csv(url, sep='\t', on_bad_lines='skip')
except Exception as e:
    print(f"Erro ao carregar com tabulação: {e}")
    try:
        # Tentativa com ponto e vírgula como delimitador
        df = pd.read_csv(url, sep=';', on_bad_lines='skip')
    except Exception as e:
        print(f"Erro ao carregar com ponto e vírgula: {e}")
        try:
            # Tentativa com vírgula como delimitador, ignorando linhas ruins
            df = pd.read_csv(url, sep=',', on_bad_lines='skip')
        except Exception as e:
            print(f"Erro ao carregar com vírgula e on_bad_lines='skip': {e}")
            # Se todas falharem, tente ler as primeiras linhas para inspeção
            try:
                import requests
                from io import StringIO
                response = requests.get(url)
                response.raise_for_status() # Check for HTTP errors
                # Read only the first few lines to inspect the format
                df = pd.read_csv(StringIO(response.text.splitlines()[0:100]), sep='\t', on_bad_lines='skip')
                print("Carregado as primeiras 100 linhas com tabulação para inspeção.")
            except Exception as e_inspect:
                 print(f"Erro final ao tentar inspecionar o arquivo: {e_inspect}")
                 df = None # Ensure df is None if all attempts fail


# Exibir as primeiras linhas do DataFrame para verificar o carregamento
if df is not None:
    display(df.head())
    print(f"\nDataFrame carregado com {len(df)} linhas.")
else:
    print("\nNão foi possível carregar o DataFrame.")

,Data,Hora,Transportadora,SM nº,Operação,CD Origem,Destino,Veículo,Local da Ocorrência,UF,...,Pronta Resposta,Solicitante do Acionamento,CD ORIGEM,Semana,Mês,ANO,UF DESTINO,CD ORIGEM,TRANSP,Condutor
0,11/01/2019,08:39:00,KILLIK - TKD,9968886,DSL,CD 666,SÃO PAULO/SP,EWJ8990,São Paulo,SP,...,Não,NaN,464,1,JAN,2019,SP,464,TRO,VALTER MARINHO CORDEIRO
1,23/01/2019,21:01:00,KILLIK - TKD,9989887,DSL,CD 666,SAO PAULO/SP,EWJ8990,ARUJA,SP,...,Não,NaN,464,3,JAN,2019,SP,464,TRO,VALTER MARINHO CORDEIRO
2,24/01/2019,08:00:00,KILLIK - TKD,9989887,DSL,CD 666,SAO PAULO/SP,EWJ8990,Sao Paulo,SP,...,Não,NaN,464,3,JAN,2019,SP,464,TRO,VALTER MARINHO CORDEIRO
3,24/01/2019,16:44:00,KILLIK - TKD,9989887,Retorno,CD 666,SAO PAULO/SP,EWJ8990,Vila Nancy,SP,...,Não,NaN,464,3,JAN,2019,SP,464,TRO,VALTER MARINHO CORDEIRO
4,13/02/2019,00:41:00,KILLIK - TKD,9899880,DSL,CD 666,DIADEMA/SP,EWJ8990,Arujá,SP,...,Não,NaN,464,6,FEV,2019,SP,464,TRO,VALTER MARINHO CORDEIRO



DataFrame carregado com 7622 linhas.


## 3. Divisão Inicial dos Dados

Nesta etapa, vamos dividir o conjunto de dados original em dois subconjuntos: um para treinamento (`df_trabalho`) e outro para validação (`df_validacao`). A separação dos dados de validação no início do projeto é uma prática recomendada para garantir que a avaliação final do modelo seja o mais realista possível, simulando a performance em dados não vistos durante o treinamento e o pré-processamento.

Utilizaremos a função `train_test_split` da biblioteca `sklearn.model_selection` para realizar essa divisão. Separaremos 20% dos dados para o conjunto de validação, mantendo 80% para o conjunto de trabalho, onde realizaremos as etapas de EDA, limpeza, engenharia de atributos e balanceamento. O conjunto de validação será mantido intacto até a fase final de avaliação do modelo.

In [ ]:
from sklearn.model_selection import train_test_split

# Dividir os dados em conjuntos de trabalho (80%) e validação (20%)
df_trabalho, df_validacao = train_test_split(df, test_size=0.2, random_state=42)

# Exibir o tamanho dos conjuntos resultantes
print(f"Tamanho do conjunto de trabalho: {len(df_trabalho)} linhas")
print(f"Tamanho do conjunto de validação: {len(df_validacao)} linhas")

Tamanho do conjunto de trabalho: 6097 linhas
Tamanho do conjunto de validação: 1525 linhas


## 4. Análise Exploratória (EDA)

Nesta etapa, vamos realizar uma análise exploratória inicial no conjunto de dados de trabalho (`df_trabalho`) para entender melhor a estrutura dos dados, identificar possíveis problemas e obter insights que serão úteis nas próximas etapas de limpeza e engenharia de atributos.

Focaremos em:
- Verificar os tipos de dados de cada coluna.
- Obter estatísticas descritivas para as variáveis numéricas, como média, desvio padrão, mínimo, máximo e quartis.

## 5. Limpeza e Imputação dos Dados

Nesta etapa, vamos identificar e tratar os valores nulos presentes no conjunto de dados de trabalho (`df_trabalho`). A presença de valores ausentes pode impactar a performance dos modelos de machine learning, e é importante lidar com eles de forma apropriada.

Primeiro, vamos verificar quais colunas contêm valores nulos e a quantidade de nulos em cada uma.

In [ ]:
# Verificar valores nulos por coluna
print("Valores nulos por coluna:")
print(df_trabalho.isnull().sum())

Valores nulos por coluna:
Data                             0
Hora                             0
Transportadora                   0
SM nº                            0
Operação                         0
CD Origem                       26
Destino                         26
Veículo                          0
Local da Ocorrência              0
UF                               0
Natureza do Evento               0
Causa Raiz                       0
Processo Macro                   0
Descrição da Ocorrência        534
Pronta Resposta                  0
Solicitante do Acionamento    5976
CD ORIGEM                        0
Semana                           0
Mês                              0
ANO                              0
UF DESTINO                      26
CD ORIGEM                        0
TRANSP                           0
Condutor                         0
dtype: int64


In [ ]:
# Verificar os tipos de dados
print("Tipos de dados:")
print(df_trabalho.dtypes)

# Estatísticas descritivas para variáveis numéricas
print("\nEstatísticas descritivas para variáveis numéricas:")
display(df_trabalho.describe())

Tipos de dados:
Data                          object
Hora                          object
Transportadora                object
SM nº                          int64
Operação                      object
CD Origem                     object
Destino                       object
Veículo                       object
Local da Ocorrência           object
UF                            object
Natureza do Evento            object
Causa Raiz                    object
Processo Macro                object
Descrição da Ocorrência       object
Pronta Resposta               object
Solicitante do Acionamento    object
CD ORIGEM                     object
Semana                         int64
Mês                           object
ANO                            int64
UF DESTINO                    object
CD ORIGEM                     object
TRANSP                        object
Condutor                      object
dtype: object

Estatísticas descritivas para variáveis numéricas:


,SM nº,Semana,ANO
count,6.097000e+03,6097.000000,6097.0
mean,9.721234e+06,25.092012,2019.0
std,1.199542e+05,11.147494,0.0
min,9.500067e+06,1.000000,2019.0
25%,9.609878e+06,17.000000,2019.0
50%,9.697659e+06,26.000000,2019.0
75%,9.789880e+06,34.000000,2019.0
max,9.999998e+06,52.000000,2019.0


## 6. Engenharia de Atributos

Nesta etapa, vamos criar novas variáveis a partir das informações existentes, especialmente da coluna 'Data', para capturar padrões que possam ser importantes para a previsão do tipo de sinistro. Também identificaremos feriados e removeremos colunas que consideramos irrelevantes para o modelo preditivo.

### Extração de Atributos da Data

Vamos extrair o dia da semana, o mês, o ano e o período do dia a partir da coluna 'Data'. Para isso, primeiro converteremos a coluna 'Data' para o tipo datetime.

In [ ]:
!pip install holidays

In [ ]:
import holidays

# Converter a coluna 'Data' para datetime
df_trabalho['Data'] = pd.to_datetime(df_trabalho['Data'], format='%d/%m/%Y')

# Extrair o dia da semana
df_trabalho['Dia_Semana'] = df_trabalho['Data'].dt.day_name()

# Extrair o mês
df_trabalho['Mês_Nome'] = df_trabalho['Data'].dt.month_name()

# Extrair o ano (já temos, mas podemos garantir o tipo)
df_trabalho['ANO'] = df_trabalho['Data'].dt.year

# Extrair o período do dia (Manhã, Tarde, Noite, Madrugada)
def periodo_do_dia(hora):
    h = int(hora.split(':')[0])
    if 5 <= h < 12:
        return 'Manhã'
    elif 12 <= h < 18:
        return 'Tarde'
    elif 18 <= h < 23:
        return 'Noite'
    else:
        return 'Madrugada'

df_trabalho['Periodo_Dia'] = df_trabalho['Hora'].apply(periodo_do_dia)

# Identificar feriados e adicionar a coluna 'É_Feriado'
# Criar um objeto de feriados para o Brasil com base nos anos presentes na coluna 'Data'
br_holidays = holidays.Brazil(years=df_trabalho['Data'].dt.year.unique())

# Função para verificar se uma data é feriado
def is_holiday(date):
    return date in br_holidays

# Aplicar a função para criar a coluna 'É_Feriado'
df_trabalho['É_Feriado'] = df_trabalho['Data'].apply(is_holiday)


print("Atributos de data e hora, incluindo 'É_Feriado', extraídos:")
display(df_trabalho[['Data', 'Dia_Semana', 'Mês_Nome', 'ANO', 'Hora', 'Periodo_Dia', 'É_Feriado']].head())

Atributos de data e hora, incluindo 'É_Feriado', extraídos:


,Data,Dia_Semana,Mês_Nome,ANO,Hora,Periodo_Dia,É_Feriado
3012,2019-02-24,Sunday,February,2019,07:34:00,Manhã,False
6210,2019-05-15,Wednesday,May,2019,16:50:00,Tarde,False
5168,2019-10-09,Wednesday,October,2019,22:12:00,Noite,False
4912,2019-08-23,Friday,August,2019,16:54:00,Tarde,False
1662,2019-07-06,Saturday,July,2019,22:48:00,Noite,False


In [ ]:
import holidays

# Criar um objeto de feriados para o Brasil no ano de 2019 (ou os anos presentes nos dados)
br_holidays = holidays.Brazil(years=df_trabalho['ANO'].unique())

# Função para verificar se uma data é feriado
def is_holiday(date):
    return date in br_holidays

# Aplicar a função para criar a coluna 'É_Feriado'
df_trabalho['É_Feriado'] = df_trabalho['Data'].apply(is_holiday)

print("\nColuna 'É_Feriado' adicionada:")
display(df_trabalho[['Data', 'Dia_Semana', 'É_Feriado']].head())


Coluna 'É_Feriado' adicionada:


,Data,Dia_Semana,É_Feriado
3012,2019-02-24,Sunday,False
6210,2019-05-15,Wednesday,False
5168,2019-10-09,Wednesday,False
4912,2019-08-23,Friday,False
1662,2019-07-06,Saturday,False


**Tratamento de Valores Nulos**

Com base na contagem de valores nulos, as colunas que precisam de tratamento são: `CD Origem`, `Destino`, `Descrição da Ocorrência`, `Solicitante do Acionamento` e `UF DESTINO`.

Vamos analisar cada uma:

*   **`CD Origem`, `Destino`, `UF DESTINO`**: Estas colunas têm um número relativamente pequeno de valores nulos (26 em 6097 linhas, aproximadamente 0.4%). Dado o baixo percentual e o fato de serem variáveis categóricas (ou que representam localizações que podem ser tratadas como categorias), a remoção das linhas com valores nulos nessas colunas é uma opção razoável. Removeremos essas linhas para manter a integridade dos dados de localização.

*   **`Descrição da Ocorrência`**: Esta coluna possui 534 valores nulos. É uma variável textual que provavelmente contém informações relevantes para a previsão do tipo de sinistro. Remover as linhas com nulos pode resultar na perda de uma quantidade considerável de dados úteis. A imputação com um valor placeholder como "sem descrição" ou a remoção seletiva (se a descrição não for crucial para um subconjunto específico de modelos) são opções. Dado que a descrição pode ser muito útil, vamos optar por preencher os valores nulos com uma string indicativa, como "Não Informado", para não perder o restante das informações da linha.

*   **`Solicitante do Acionamento`**: Esta coluna tem um número muito alto de valores nulos (5976 em 6097 linhas, aproximadamente 98%). Com uma quantidade tão esmagadora de dados ausentes, esta coluna provavelmente não agregará valor preditivo e pode até introduzir ruído. A melhor abordagem aqui é remover completamente a coluna.

In [ ]:
# Tratamento de valores nulos no df_trabalho

# Remover linhas com nulos em 'CD Origem', 'Destino', 'UF DESTINO'
df_trabalho = df_trabalho.dropna(subset=['CD Origem', 'Destino', 'UF DESTINO'])
print(f"Tamanho do conjunto de trabalho após remover nulos em localização: {len(df_trabalho)} linhas")

# Preencher valores nulos em 'Descrição da Ocorrência'
df_trabalho['Descrição da Ocorrência'] = df_trabalho['Descrição da Ocorrência'].fillna('Não Informado')
print(f"Valores nulos em 'Descrição da Ocorrência' após preenchimento: {df_trabalho['Descrição da Ocorrência'].isnull().sum()}")

# Verificar novamente valores nulos para confirmar
print("\nValores nulos restantes por coluna:")
print(df_trabalho.isnull().sum())

Tamanho do conjunto de trabalho após remover nulos em localização: 6071 linhas
Valores nulos em 'Descrição da Ocorrência' após preenchimento: 0

Valores nulos restantes por coluna:
Data                             0
Hora                             0
Transportadora                   0
SM nº                            0
Operação                         0
CD Origem                        0
Destino                          0
Veículo                          0
Local da Ocorrência              0
UF                               0
Natureza do Evento               0
Causa Raiz                       0
Processo Macro                   0
Descrição da Ocorrência          0
Pronta Resposta                  0
Solicitante do Acionamento    5950
CD ORIGEM                        0
Semana                           0
Mês                              0
ANO                              0
UF DESTINO                       0
CD ORIGEM                        0
TRANSP                           0
Condutor      

## 7. Balanceamento

Nesta etapa, vamos avaliar se o conjunto de dados de trabalho (`df_trabalho`) apresenta um desbalanceamento significativo na variável alvo. O desbalanceamento ocorre quando uma ou mais classes da variável alvo têm muito menos instâncias do que outras. Isso pode levar modelos de machine learning a serem tendenciosos para a classe majoritária, resultando em baixa performance na previsão das classes minoritárias.

Primeiro, precisamos definir qual coluna será a variável alvo (`y`) para a previsão. Com base na descrição do problema, as colunas `Natureza do Evento`, `Causa Raiz` ou `Processo Macro` são candidatas prováveis para representar o "tipo de sinistro". Para este pipeline, vamos considerar `Natureza do Evento` como a variável alvo principal, mas a abordagem seria similar se outra coluna fosse escolhida.

Vamos verificar a distribuição das classes na coluna `Natureza do Evento`.

In [ ]:
# Definir a variável alvo (escolhendo 'Natureza do Evento' como exemplo)
target_column = 'Natureza do Evento'

# Verificar a distribuição das classes na variável alvo
print(f"Distribuição das classes na coluna '{target_column}':")
display(df_trabalho[target_column].value_counts())

# Calcular o percentual de cada classe
print(f"\nPercentual da distribuição das classes na coluna '{target_column}':")
display(df_trabalho[target_column].value_counts(normalize=True) * 100)

# Avaliar se há necessidade de balanceamento (geralmente, se a classe minoritária for < 5-10%)
# Neste caso, há muitas classes, e a distribuição parece bastante desigual, indicando a necessidade de balanceamento.

Distribuição das classes na coluna 'Natureza do Evento':


,count
Natureza do Evento,
Excesso de velocidade,1407
Parada Irregular,927
Tempo de Parada Excedido,519
Desvio de Rota,370
Inicio de viagem sem aguardar a mensagem OK XPTO,318
Violação de porta de cabine,242
Jammer,238
Perda de Sinal,197
Problema Mecânico,195



Percentual da distribuição das classes na coluna 'Natureza do Evento':


,proportion
Natureza do Evento,
Excesso de velocidade,23.175754
Parada Irregular,15.269313
Tempo de Parada Excedido,8.548839
Desvio de Rota,6.094548
Inicio de viagem sem aguardar a mensagem OK XPTO,5.238017
Violação de porta de cabine,3.986164
Jammer,3.920277
Perda de Sinal,3.244935
Problema Mecânico,3.211991


### Exclusão de Variáveis Irrelevantes

Algumas colunas no conjunto de dados podem não ser relevantes para a previsão do tipo de sinistro ou podem conter informações redundantes ou de identificação que não são úteis para o modelo. Vamos revisar as colunas existentes e decidir quais remover, justificando a decisão.

Com base na análise inicial e no objetivo de prever o *tipo* de sinistro (que provavelmente está relacionado à `Natureza do Evento`, `Causa Raiz`, `Processo Macro`, `Descrição da Ocorrência` e talvez `Pronta Resposta`), as seguintes colunas podem ser candidatas à remoção:

*   **`SM nº`**: Número identificador único da ocorrência. Não deve ter valor preditivo para o *tipo* de sinistro em si.
*   **`Transportadora`**: Embora possa haver padrões específicos por transportadora, para um modelo geral que possa ser aplicado a diferentes transportadoras, essa variável pode não ser ideal ou exigiria um tratamento mais complexo (como one-hot encoding com muitas categorias). Vamos removê-la por enquanto para simplificar.
*   **`Veículo`**: Identificador do veículo. Similar ao `SM nº`, é um identificador e não deve ter valor preditivo direto para o tipo de sinistro.
*   **`Local da Ocorrência`**: Informação de texto livre sobre o local. Pode ser útil, mas a `UF` e o `CD Origem`/`Destino` já fornecem informações de localização estruturadas. Removeremos para simplificar e evitar a complexidade de processamento de texto neste momento.
*   **`UF`**: Já temos `UF DESTINO`, que pode ser mais relevante para o destino final da carga. Vamos manter `UF DESTINO` e remover `UF`.
*   **`CD ORIGEM` (as duplicadas)**: Parece haver colunas `CD Origem` duplicadas. Manteremos apenas uma versão.
*   **`Semana`**: Já extraímos o `Dia_Semana` e o `Mês_Nome`. A `Semana` pode ser redundante ou menos informativa que o dia da semana e o mês para capturar padrões sazonais ou de frequência.
*   **`Mês`**: Já extraímos o `Mês_Nome`, que é mais descritivo.
*   **`ANO`**: Como todos os dados parecem ser de 2019, esta coluna não tem variabilidade e não agregará valor preditivo.
*   **`TRANSP`**: Parece ser um código da transportadora, redundante com a coluna `Transportadora` que planejamos remover.
*   **`Condutor`**: Nome do condutor. Identificador pessoal, não deve ter valor preditivo para o tipo de sinistro.
*   **`Data` e `Hora`**: Já extraímos atributos relevantes dessas colunas (`Dia_Semana`, `Mês_Nome`, `Periodo_Dia`, `É_Feriado`). Manter as colunas originais pode ser redundante e o formato pode não ser ideal para alguns modelos.

Manteremos as seguintes colunas que parecem mais relevantes para a previsão do *tipo* de sinistro: `Operação`, `CD Origem` (uma versão), `Destino`, `UF DESTINO`, `Natureza do Evento` (potencial alvo), `Causa Raiz` (potencial alvo/feature), `Processo Macro` (potencial alvo/feature), `Descrição da Ocorrência`, `Pronta Resposta`, `Dia_Semana`, `Mês_Nome`, `Periodo_Dia`, `É_Feriado`.

**Nota**: A variável alvo (`y`) ainda precisa ser definida formalmente. `Natureza do Evento`, `Causa Raiz` ou `Processo Macro` são candidatas prováveis. Por enquanto, vamos manter todas elas como possíveis features até a etapa de modelagem.

In [ ]:
# Colunas a serem removidas com justificativa
colunas_para_remover = [
    'SM nº',        # Identificador único, sem valor preditivo para o tipo
    'Transportadora', # Pode ser muito categórica/específica, simplificando por agora
    'Veículo',      # Identificador, sem valor preditivo direto para o tipo
    'Local da Ocorrência', # Redundante com UF/CD Origem/Destino e mais complexo de processar
    'UF',           # Redundante com UF DESTINO
    'Semana',       # Redundante com Dia_Semana e Mês_Nome
    'Mês',          # Redundante com Mês_Nome
    'ANO',          # Sem variabilidade (apenas 2019)
    'TRANSP',       # Código da transportadora, redundante
    'Condutor',     # Identificador pessoal, sem valor preditivo para o tipo
    'Data',         # Atributos extraídos, coluna original não necessária
    'Hora'          # Atributos extraídos, coluna original não necessária
]

# Identificar colunas duplicadas de 'CD Origem' e remover as extras
colunas_cd_origem = [col for col in df_trabalho.columns if 'CD ORIGEM' in col.upper()]
# Manter apenas a primeira ocorrência e remover as demais
if len(colunas_cd_origem) > 1:
    colunas_para_remover.extend(colunas_cd_origem[1:])
    print(f"Colunas duplicadas de CD ORIGEM identificadas e adicionadas para remoção: {colunas_cd_origem[1:]}")

# Remover as colunas especificadas
df_trabalho = df_trabalho.drop(columns=colunas_para_remover, errors='ignore') # errors='ignore' para não falhar se a coluna já foi removida

print(f"\nColunas restantes após remoção: {df_trabalho.columns.tolist()}")
print(f"Tamanho do conjunto de trabalho após remoção de colunas: {df_trabalho.shape}")

# Realizar EDA nos dados resultantes (verificar tipos e estatísticas numéricas novamente)
print("\nTipos de dados após engenharia de atributos e remoção de colunas:")
print(df_trabalho.dtypes)

print("\nEstatísticas descritivas para variáveis numéricas após engenharia de atributos e remoção de colunas:")
display(df_trabalho.describe())

print("\nPrimeiras linhas do DataFrame após engenharia de atributos e remoção de colunas:")
display(df_trabalho.head())

Colunas duplicadas de CD ORIGEM identificadas e adicionadas para remoção: ['CD ORIGEM ', 'CD ORIGEM']

Colunas restantes após remoção: ['Operação', 'CD Origem', 'Destino', 'Natureza do Evento', 'Causa Raiz', 'Processo Macro', 'Descrição da Ocorrência', 'Pronta Resposta', 'Solicitante do Acionamento', 'UF DESTINO', 'Dia_Semana', 'Mês_Nome', 'Periodo_Dia', 'É_Feriado']
Tamanho do conjunto de trabalho após remoção de colunas: (6071, 14)

Tipos de dados após engenharia de atributos e remoção de colunas:
Operação                      object
CD Origem                     object
Destino                       object
Natureza do Evento            object
Causa Raiz                    object
Processo Macro                object
Descrição da Ocorrência       object
Pronta Resposta               object
Solicitante do Acionamento    object
UF DESTINO                    object
Dia_Semana                    object
Mês_Nome                      object
Periodo_Dia                   object
É_Feriado     

,Operação,CD Origem,Destino,Natureza do Evento,Causa Raiz,Processo Macro,Descrição da Ocorrência,Pronta Resposta,Solicitante do Acionamento,UF DESTINO,Dia_Semana,Mês_Nome,Periodo_Dia,É_Feriado
count,6071,6071,6071,6071,6071,6071,6071,6071,121,6071,6071,6071,6071,6071
unique,5,43,157,55,20,14,3493,3,28,27,7,11,4,2
top,DSL,CD 116,RIO DE JANEIRO/RJ,Excesso de velocidade,Desvio operacional,Falha do Condutor,Não Informado,Não,Automatico,SP,Wednesday,August,Tarde,False
freq,4036,3240,1076,1407,1802,2858,529,5925,41,1312,1100,1084,2142,5969



Primeiras linhas do DataFrame após engenharia de atributos e remoção de colunas:


,Operação,CD Origem,Destino,Natureza do Evento,Causa Raiz,Processo Macro,Descrição da Ocorrência,Pronta Resposta,Solicitante do Acionamento,UF DESTINO,Dia_Semana,Mês_Nome,Periodo_Dia,É_Feriado
3012,DSL,CD 116,MOSSORO/RN,Violação de Engate,Violação de Equipamento,Manutenção,Veículo gerou desengate e engate de carreta 2.,Não,NaN,RN,Sunday,February,Manhã,False
6210,DSL,CD 116,TERESINA/PI,Tempo de Parada Excedido,Desvio operacional,Falha do Condutor,VEICULO EM PARADA EXCEDIDA COM INFORME PARA AB...,Não,NaN,PI,Wednesday,May,Tarde,False
5168,TSF,CD 326,ARUJA/SP,Tempo de Parada Excedido,Desvio operacional,Falha do Condutor,Veiculo em parada excedida com informe para ab...,Não,NaN,SP,Wednesday,October,Noite,False
4912,Retorno,XD 666,JAU/SP,Parada Irregular,Desvio operacional,Falha do Condutor,CONDUTOR REALIZOU PARADA EM LOCALIDADE PROIBIDA,Não,NaN,SP,Friday,August,Tarde,False
1662,TSF,XD 666,SÃO JOSE/SC,Violação de Baú,Violação de Equipamento,Manutenção,VEICULO GEROU BAU ABERTO,Não,NaN,SC,Saturday,July,Noite,False


# Task
Aplicar SMOTE para balancear o conjunto de dados de trabalho.

## Identificar e codificar variáveis categóricas

### Subtask:
Identificar as colunas categóricas restantes no conjunto de trabalho e aplicar codificação (por exemplo, One-Hot Encoding) para prepará-las para o SMOTE.


**Reasoning**:
Identify object type columns, apply one-hot encoding to them, and concatenate the results with the non-object columns.



In [ ]:
# Identificar colunas categóricas (tipo 'object')
categorical_cols = df_trabalho.select_dtypes(include='object').columns
print(f"Colunas categóricas identificadas para One-Hot Encoding: {categorical_cols.tolist()}")

# Excluir a coluna 'Solicitante do Acionamento' se ela ainda estiver presente e tiver muitos nulos,
# conforme decidido na etapa de limpeza, antes de aplicar One-Hot Encoding.
# Embora a contagem de nulos seja mostrada no passo anterior, a coluna não foi removida ainda.
if 'Solicitante do Acionamento' in categorical_cols:
    print("Removendo a coluna 'Solicitante do Acionamento' devido ao alto número de valores nulos.")
    df_trabalho = df_trabalho.drop(columns=['Solicitante do Acionamento'])
    categorical_cols = df_trabalho.select_dtypes(include='object').columns
    print(f"Colunas categóricas restantes: {categorical_cols.tolist()}")


# Aplicar One-Hot Encoding nas colunas categóricas restantes
df_trabalho_encoded = pd.get_dummies(df_trabalho, columns=categorical_cols, drop_first=True)

# 'É_Feriado' é booleana e já está em formato numérico (True/False), converter para int (0/1)
df_trabalho_encoded['É_Feriado'] = df_trabalho_encoded['É_Feriado'].astype(int)


print("\nDataFrame após One-Hot Encoding:")
display(df_trabalho_encoded.head())
print(f"\nTamanho do DataFrame após One-Hot Encoding: {df_trabalho_encoded.shape}")
print("\nTipos de dados após One-Hot Encoding:")
print(df_trabalho_encoded.dtypes)

Colunas categóricas identificadas para One-Hot Encoding: ['Operação', 'CD Origem', 'Destino', 'Natureza do Evento', 'Causa Raiz', 'Processo Macro', 'Descrição da Ocorrência', 'Pronta Resposta', 'Solicitante do Acionamento', 'UF DESTINO', 'Dia_Semana', 'Mês_Nome', 'Periodo_Dia']
Removendo a coluna 'Solicitante do Acionamento' devido ao alto número de valores nulos.
Colunas categóricas restantes: ['Operação', 'CD Origem', 'Destino', 'Natureza do Evento', 'Causa Raiz', 'Processo Macro', 'Descrição da Ocorrência', 'Pronta Resposta', 'UF DESTINO', 'Dia_Semana', 'Mês_Nome', 'Periodo_Dia']

DataFrame após One-Hot Encoding:


,É_Feriado,Operação_Exportação,Operação_Importação,Operação_Retorno,Operação_TSF,CD Origem_BASE BRIX SANTAREM,CD Origem_BASE BRIX/POA MACAPA,CD Origem_BRIX MAO,CD Origem_Base Sequoia,CD Origem_CD 116,...,Mês_Nome_July,Mês_Nome_June,Mês_Nome_March,Mês_Nome_May,Mês_Nome_November,Mês_Nome_October,Mês_Nome_September,Periodo_Dia_Manhã,Periodo_Dia_Noite,Periodo_Dia_Tarde
3012,0,False,False,False,False,False,False,False,False,True,...,False,False,False,False,False,False,False,True,False,False
6210,0,False,False,False,False,False,False,False,False,True,...,False,False,False,True,False,False,False,False,False,True
5168,0,False,False,False,True,False,False,False,False,False,...,False,False,False,False,False,True,False,False,True,False
4912,0,False,False,True,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,True
1662,0,False,False,False,True,False,False,False,False,False,...,True,False,False,False,False,False,False,False,True,False



Tamanho do DataFrame após One-Hot Encoding: (6071, 3828)

Tipos de dados após One-Hot Encoding:
É_Feriado              int64
Operação_Exportação     bool
Operação_Importação     bool
Operação_Retorno        bool
Operação_TSF            bool
                       ...  
Mês_Nome_October        bool
Mês_Nome_September      bool
Periodo_Dia_Manhã       bool
Periodo_Dia_Noite       bool
Periodo_Dia_Tarde       bool
Length: 3828, dtype: object


### Subtask:
Aplicar a técnica SMOTE (ou uma variação apropriada para dados mistos, se aplicável) no conjunto de treinamento (X_trabalho, y_trabalho) para aumentar o número de instâncias das classes minoritárias.

In [ ]:
!pip install imbalanced-learn

In [ ]:
# Definir a variável alvo (y) e as features (X)
# A coluna alvo é 'Natureza do Evento', que foi one-hot encoded.
# Precisamos identificar as colunas que correspondem à variável alvo codificada.
target_encoded_columns = [col for col in df_trabalho_encoded.columns if col.startswith(f'{target_column}_')]

# Separar features (X) e variável alvo (y)
X_trabalho = df_trabalho_encoded.drop(columns=target_encoded_columns)
y_trabalho = df_trabalho_encoded[target_encoded_columns]

print("Conjunto de trabalho separado em features (X_trabalho) e variável alvo (y_trabalho).")
print(f"Formato de X_trabalho: {X_trabalho.shape}")
print(f"Formato de y_trabalho: {y_trabalho.shape}")

# Exibir as primeiras linhas de X_trabalho e y_trabalho para verificar
print("\nPrimeiras linhas de X_trabalho:")
display(X_trabalho.head())

print("\nPrimeiras linhas de y_trabalho:")
display(y_trabalho.head())

Conjunto de trabalho separado em features (X_trabalho) e variável alvo (y_trabalho).
Formato de X_trabalho: (6071, 3774)
Formato de y_trabalho: (6071, 54)

Primeiras linhas de X_trabalho:


,É_Feriado,Operação_Exportação,Operação_Importação,Operação_Retorno,Operação_TSF,CD Origem_BASE BRIX SANTAREM,CD Origem_BASE BRIX/POA MACAPA,CD Origem_BRIX MAO,CD Origem_Base Sequoia,CD Origem_CD 116,...,Mês_Nome_July,Mês_Nome_June,Mês_Nome_March,Mês_Nome_May,Mês_Nome_November,Mês_Nome_October,Mês_Nome_September,Periodo_Dia_Manhã,Periodo_Dia_Noite,Periodo_Dia_Tarde
3012,0,False,False,False,False,False,False,False,False,True,...,False,False,False,False,False,False,False,True,False,False
6210,0,False,False,False,False,False,False,False,False,True,...,False,False,False,True,False,False,False,False,False,True
5168,0,False,False,False,True,False,False,False,False,False,...,False,False,False,False,False,True,False,False,True,False
4912,0,False,False,True,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,True
1662,0,False,False,False,True,False,False,False,False,False,...,True,False,False,False,False,False,False,False,True,False



Primeiras linhas de y_trabalho:


,Natureza do Evento_Bloqueio em área de risco,Natureza do Evento_Botão de Risco,Natureza do Evento_Chave geral desligada,Natureza do Evento_Chegada não informada,Natureza do Evento_Cliente pediu Retirada de Inteligencia Embarcada (Citar em Observação nome do Solicitante),Natureza do Evento_Condutor não envia mensagem,Natureza do Evento_Defeito na antena GPS,Natureza do Evento_Defeito na trava do baú,Natureza do Evento_Defeito no Bloqueador,Natureza do Evento_Defeito no Computador de Bordo,...,Natureza do Evento_Uso Excessivo de Macro,Natureza do Evento_Veiculo não espelhado,Natureza do Evento_Viagem realizada com defeito no dispositivo,Natureza do Evento_Violação de Baú,Natureza do Evento_Violação de Engate,Natureza do Evento_Violação de Painel,Natureza do Evento_Violação de antena,Natureza do Evento_Violação de bateria,Natureza do Evento_Violação de porta de cabine,Natureza do Evento_Violação do cabo da antena
3012,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,True,False,False,False,False,False
6210,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
5168,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4912,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1662,False,False,False,False,False,False,False,False,False,False,...,False,False,False,True,False,False,False,False,False,False


In [ ]:
from imblearn.over_sampling import SMOTE
import pandas as pd
import numpy as np

# Como temos dados mistos (numéricos e categóricos one-hot encoded), o SMOTE padrão pode não ser ideal.
# No entanto, como todas as colunas categóricas foram one-hot encoded para valores numéricos (0 e 1),
# o SMOTE padrão deve funcionar. Para datasets com variáveis categóricas que não foram OHE,
# seria mais apropriado usar SMOTENC (SMOTE for Nominal and Continuous).
# Dado o formato atual de X_trabalho, o SMOTE padrão é aplicável.

# Definir um valor para k_neighbors para o SMOTE
n_neighbors_smote = 1

# Inicializar o SMOTE
# Usar sampling_strategy='not majority' para focar nas classes minoritárias
# random_state para reprodutibilidade
# Ajustar k_neighbors para lidar com classes com poucas amostras
smote = SMOTE(sampling_strategy='not majority', random_state=42, k_neighbors=n_neighbors_smote)

print(f"Formato original de X_trabalho: {X_trabalho.shape}, y_trabalho: {y_trabalho.shape}")
print(f"Aplicando SMOTE com sampling_strategy='not majority' e k_neighbors={n_neighbors_smote}...")

# Aplicar SMOTE no conjunto de trabalho
# y_trabalho é um DataFrame com colunas one-hot encoded. SMOTE requer um array 1D para o target.
# Precisamos converter y_trabalho para um formato que o SMOTE entenda.
# Uma maneira é encontrar o índice da coluna True para cada linha (representando a classe)

# Encontrar o índice da coluna com valor True para cada linha em y_trabalho
# idxmax(axis=1) retorna o nome da coluna com o valor máximo (True=1, False=0)
y_trabalho_labels = y_trabalho.idxmax(axis=1)

# SMOTE não funciona diretamente com nomes de colunas como labels.
# Precisamos converter esses nomes de colunas de volta para representações numéricas ou rótulos simples.
# Vamos mapear cada nome de coluna para um rótulo numérico único.
# Primeiro, obter todas as classes únicas (nomes das colunas de y_trabalho)
classes = y_trabalho.columns.tolist()
# Criar um mapeamento de nome da classe para índice numérico
class_mapping = {class_name: i for i, class_name in enumerate(classes)}
# Mapear os rótulos das linhas para os índices numéricos
y_trabalho_numeric = y_trabalho_labels.map(class_mapping)

# --- Adição para tratar classes com menos de 2 amostras ---
# Identificar as classes com contagem menor que 2
class_counts = pd.Series(y_trabalho_numeric).value_counts()
classes_to_remove = class_counts[class_counts < 2].index.tolist()

if classes_to_remove:
    print(f"Classes com menos de 2 instâncias (serão removidas antes do SMOTE): {classes_to_remove}")
    # Identificar os índices das linhas que pertencem a essas classes
    indices_to_remove = y_trabalho_numeric[y_trabalho_numeric.isin(classes_to_remove)].index

    # Remover essas linhas de X_trabalho e y_trabalho_numeric
    X_trabalho_filtered = X_trabalho.drop(index=indices_to_remove)
    y_trabalho_numeric_filtered = y_trabalho_numeric.drop(index=indices_to_remove)

    print(f"Formato de X_trabalho após remover classes com menos de 2 instâncias: {X_trabalho_filtered.shape}")
    print(f"Formato de y_trabalho_numeric após remover classes com menos de 2 instâncias: {y_trabalho_numeric_filtered.shape}")

    # Usar os dados filtrados para o SMOTE
    X_apply_smote = X_trabalho_filtered
    y_apply_smote = y_trabalho_numeric_filtered
else:
    # Se não houver classes com menos de 2 instâncias, usar os dados originais
    X_apply_smote = X_trabalho
    y_apply_smote = y_trabalho_numeric
# --- Fim da adição ---


# Agora aplicamos SMOTE com os dados numéricos e os rótulos numéricos (filtrados, se necessário)
X_res, y_res_numeric = smote.fit_resample(X_apply_smote, y_apply_smote)

print("SMOTE aplicado.")
print(f"Formato de X_res (features rebalanceadas): {X_res.shape}")
print(f"Formato de y_res_numeric (variável alvo rebalanceada - numérica): {y_res_numeric.shape}")

# Para manter a consistência, podemos converter y_res_numeric de volta para one-hot encoding,
# mas isso pode gerar um DataFrame muito grande e esparso.
# Para visualização do balanceamento, podemos usar value_counts em y_res_numeric.
# Para modelagem, a representação numérica de y_res_numeric geralmente é suficiente,
# e os modelos podem lidar com classificação multiclasse com rótulos numéricos.

# Opcional: Converter y_res_numeric de volta para nomes de classes para visualização
# Criar um mapeamento inverso
inverse_class_mapping = {i: class_name for class_name, i in class_mapping.items()}
y_res_labels = y_res_numeric.map(inverse_class_mapping)

print("\nDistribuição das classes na variável alvo após SMOTE (rótulos numéricos):")
display(pd.Series(y_res_numeric).value_counts().sort_index())

print("\nDistribuição das classes na variável alvo após SMOTE (nomes das classes):")
display(y_res_labels.value_counts())

Formato original de X_trabalho: (6071, 3774), y_trabalho: (6071, 54)
Aplicando SMOTE com sampling_strategy='not majority' e k_neighbors=1...
Classes com menos de 2 instâncias (serão removidas antes do SMOTE): [2, 4]
Formato de X_trabalho após remover classes com menos de 2 instâncias: (6069, 3774)
Formato de y_trabalho_numeric após remover classes com menos de 2 instâncias: (6069,)
SMOTE aplicado.
Formato de X_res (features rebalanceadas): (73164, 3774)
Formato de y_res_numeric (variável alvo rebalanceada - numérica): (73164,)

Distribuição das classes na variável alvo após SMOTE (rótulos numéricos):


,count
0,1407
1,1407
3,1407
5,1407
6,1407
7,1407
8,1407
9,1407
10,1407
11,1407



Distribuição das classes na variável alvo após SMOTE (nomes das classes):


,count
Natureza do Evento_Violação de Engate,1407
Natureza do Evento_Tempo de Parada Excedido,1407
Natureza do Evento_Parada Irregular,1407
Natureza do Evento_Violação de Baú,1407
Natureza do Evento_Problema Mecânico,1407
Natureza do Evento_Excesso de velocidade,1407
Natureza do Evento_Violação de porta de cabine,1407
Natureza do Evento_Condutor não envia mensagem,1407
Natureza do Evento_Botão de Risco,1407
Natureza do Evento_Desvio de Rota,1407


**Reasoning**:
The error indicates that the `df_trabalho` DataFrame is not defined. This is likely due to a kernel restart or a previous error that prevented the DataFrame from being created. I need to recreate the `df_trabalho` DataFrame by re-executing the necessary steps from the beginning of the pipeline up to the point where `df_trabalho` is defined and processed.



# Task
Analyze the trend of content added to Netflix over time using the dataset provided in "/tmp/netflix_titles.csv".

## Data loading

### Subtask:
Load the data from "/tmp/netflix_titles.csv" into a dataframe.


**Reasoning**:
The subtask is to load the data into a pandas DataFrame and display the first few rows. This can be achieved by importing pandas, reading the CSV file, and then using the head() method.



# Task
Treinar e avaliar modelos de classificação no conjunto de trabalho balanceado e no conjunto de validação pré-processado.

## Seleção de modelos

### Subtask:
Escolher um ou mais modelos de classificação adequados para o problema (por exemplo, Random Forest, Gradient Boosting, Regressão Logística, etc.).


**Reasoning**:
Import the necessary classification model classes from scikit-learn as instructed.



In [ ]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB

print("Classification model classes imported successfully.")

Classification model classes imported successfully.


## Pré-processamento do conjunto de validação

### Subtask:
Aplicar as mesmas etapas de pré-processamento (limpeza, engenharia de atributos, codificação) ao conjunto de validação (`df_validacao`) que foram aplicadas ao conjunto de trabalho.


**Reasoning**:
Apply the same preprocessing steps applied to the training data to the validation data, including creating copies, handling datetime, extracting features, handling nulls, removing columns, and one-hot encoding.



In [ ]:
# 1. Crie cópias do df_validacao original
df_validacao_processed = df_validacao.copy()

# 2. Converta a coluna 'Data' para datetime
df_validacao_processed['Data'] = pd.to_datetime(df_validacao_processed['Data'], format='%d/%m/%Y')

# 3. Extraia os atributos de data e hora, incluindo 'É_Feriado'
df_validacao_processed['Dia_Semana'] = df_validacao_processed['Data'].dt.day_name()
df_validacao_processed['Mês_Nome'] = df_validacao_processed['Data'].dt.month_name()
df_validacao_processed['ANO'] = df_validacao_processed['Data'].dt.year # Ensure consistent type if needed
df_validacao_processed['Periodo_Dia'] = df_validacao_processed['Hora'].apply(periodo_do_dia) # Use the same function defined earlier

# Assuming br_holidays was created in a previous cell and is available
# If not, recreate it based on the years in the validation set
if 'br_holidays' not in locals():
    br_holidays = holidays.Brazil(years=df_validacao_processed['ANO'].unique())

df_validacao_processed['É_Feriado'] = df_validacao_processed['Data'].apply(is_holiday) # Use the same function defined earlier

print("Atributos de data e hora, incluindo 'É_Feriado', extraídos para o conjunto de validação.")


# 4. Remova linhas com nulos em 'CD Origem', 'Destino', 'UF DESTINO'
df_validacao_processed = df_validacao_processed.dropna(subset=['CD Origem', 'Destino', 'UF DESTINO'])
print(f"Tamanho do conjunto de validação após remover nulos em localização: {len(df_validacao_processed)} linhas")

# 5. Preencha valores nulos em 'Descrição da Ocorrência'
df_validacao_processed['Descrição da Ocorrência'] = df_validacao_processed['Descrição da Ocorrência'].fillna('Não Informado')
print(f"Valores nulos em 'Descrição da Ocorrência' após preenchimento no conjunto de validação: {df_validacao_processed['Descrição da Ocorrência'].isnull().sum()}")


# 6. Remova as mesmas colunas removidas do conjunto de trabalho
# Ensure 'colunas_para_remover' is defined from the training data preprocessing steps
# If not, redefine it based on the previous logic
if 'colunas_para_remover' not in locals():
    colunas_para_remover = [
        'SM nº', 'Transportadora', 'Veículo', 'Local da Ocorrência', 'UF',
        'Semana', 'Mês', 'ANO', 'TRANSP', 'Condutor', 'Data', 'Hora'
    ]
    # Also handle potential duplicate 'CD ORIGEM' columns in the validation set
    colunas_cd_origem_validacao = [col for col in df_validacao_processed.columns if 'CD ORIGEM' in col.upper()]
    if len(colunas_cd_origem_validacao) > 1:
        colunas_para_remover.extend(colunas_cd_origem_validacao[1:])
        print(f"Colunas duplicadas de CD ORIGEM identificadas e adicionadas para remoção no conjunto de validação: {colunas_cd_origem_validacao[1:]}")

# Excluir a coluna 'Solicitante do Acionamento' se ela ainda estiver presente e tiver muitos nulos
if 'Solicitante do Acionamento' in df_validacao_processed.columns:
     print("Removendo a coluna 'Solicitante do Acionamento' do conjunto de validação devido ao alto número de valores nulos.")
     colunas_para_remover.append('Solicitante do Acionamento')


df_validacao_processed = df_validacao_processed.drop(columns=colunas_para_remover, errors='ignore')

print(f"\nColunas restantes no conjunto de validação após remoção: {df_validacao_processed.columns.tolist()}")
print(f"Tamanho do conjunto de validação após remoção de colunas: {df_validacao_processed.shape}")

# 7. Aplicar One-Hot Encoding nas colunas categóricas restantes
categorical_cols_validacao = df_validacao_processed.select_dtypes(include='object').columns
print(f"\nColunas categóricas identificadas para One-Hot Encoding no conjunto de validação: {categorical_cols_validacao.tolist()}")


df_validacao_encoded = pd.get_dummies(df_validacao_processed, columns=categorical_cols_validacao, drop_first=True)

# 8. Converter a coluna 'É_Feriado' para int
df_validacao_encoded['É_Feriado'] = df_validacao_encoded['É_Feriado'].astype(int)

print("\nDataFrame de validação após One-Hot Encoding:")
display(df_validacao_encoded.head())
print(f"\nTamanho do DataFrame de validação após One-Hot Encoding: {df_validacao_encoded.shape}")
print("\nTipos de dados do conjunto de validação após One-Hot Encoding:")
print(df_validacao_encoded.dtypes)

Atributos de data e hora, incluindo 'É_Feriado', extraídos para o conjunto de validação.
Tamanho do conjunto de validação após remover nulos em localização: 1520 linhas
Valores nulos em 'Descrição da Ocorrência' após preenchimento no conjunto de validação: 0
Removendo a coluna 'Solicitante do Acionamento' do conjunto de validação devido ao alto número de valores nulos.

Colunas restantes no conjunto de validação após remoção: ['Operação', 'CD Origem', 'Destino', 'Natureza do Evento', 'Causa Raiz', 'Processo Macro', 'Descrição da Ocorrência', 'Pronta Resposta', 'UF DESTINO', 'Dia_Semana', 'Mês_Nome', 'Periodo_Dia', 'É_Feriado']
Tamanho do conjunto de validação após remoção de colunas: (1520, 13)

Colunas categóricas identificadas para One-Hot Encoding no conjunto de validação: ['Operação', 'CD Origem', 'Destino', 'Natureza do Evento', 'Causa Raiz', 'Processo Macro', 'Descrição da Ocorrência', 'Pronta Resposta', 'UF DESTINO', 'Dia_Semana', 'Mês_Nome', 'Periodo_Dia']

DataFrame de validaç

,É_Feriado,Operação_Exportação,Operação_Retorno,Operação_TSF,CD Origem_BASE BRIX/POA MACAPA,CD Origem_CD 116,CD Origem_CD 116 DSL Ananindeu,CD Origem_CD 116 DSL FORTALEZA,CD Origem_CD 116 DSL J. PESSOA,CD Origem_CD 116 DSL Posto 13,...,Mês_Nome_July,Mês_Nome_June,Mês_Nome_March,Mês_Nome_May,Mês_Nome_November,Mês_Nome_October,Mês_Nome_September,Periodo_Dia_Manhã,Periodo_Dia_Noite,Periodo_Dia_Tarde
1321,1,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,True,False
3520,0,False,False,False,False,True,False,False,False,False,...,False,False,True,False,False,False,False,True,False,False
3176,0,False,False,False,False,True,False,False,False,False,...,True,False,False,False,False,False,False,False,True,False
132,0,False,False,False,False,False,False,False,False,False,...,True,False,False,False,False,False,False,False,False,True
1430,0,False,False,False,False,True,False,False,False,False,...,False,False,True,False,False,False,False,False,False,True



Tamanho do DataFrame de validação após One-Hot Encoding: (1520, 1258)

Tipos de dados do conjunto de validação após One-Hot Encoding:
É_Feriado                         int64
Operação_Exportação                bool
Operação_Retorno                   bool
Operação_TSF                       bool
CD Origem_BASE BRIX/POA MACAPA     bool
                                  ...  
Mês_Nome_October                   bool
Mês_Nome_September                 bool
Periodo_Dia_Manhã                  bool
Periodo_Dia_Noite                  bool
Periodo_Dia_Tarde                  bool
Length: 1258, dtype: object


**Reasoning**:
Align the columns of the encoded validation DataFrame with the columns of the encoded training DataFrame to ensure consistency, then separate the features and the target variable in the validation set, converting the target to numerical labels using the previously created mapping.



In [ ]:
# 9. Alinhar colunas do conjunto de validação com o conjunto de trabalho
# Precisamos obter as colunas de X_apply_smote (ou X_trabalho_encoded antes da separação X/y)
# X_apply_smote é o DataFrame de features do conjunto de trabalho após OHE e antes do SMOTE
# Vamos usar X_apply_smote.columns para garantir que temos apenas as colunas de features
# e não as colunas one-hot encoded da variável alvo.

# Identificar as colunas de features no conjunto de trabalho (após OHE, antes SMOTE)
# As colunas de features são todas as colunas em df_trabalho_encoded, exceto as colunas alvo
feature_cols_trabalho = X_apply_smote.columns # X_apply_smote já tem as colunas de features

# Reindexar o DataFrame de validação codificado para ter as mesmas colunas do conjunto de trabalho (features)
# Adicionar colunas ausentes com valor 0 e manter apenas as colunas presentes no conjunto de trabalho
df_validacao_aligned = df_validacao_encoded.reindex(columns=feature_cols_trabalho, fill_value=0)

print("\nDataFrame de validação alinhado com as colunas de features do conjunto de trabalho.")
print(f"Tamanho do DataFrame de validação alinhado: {df_validacao_aligned.shape}")

# 10. Separar features (X_validacao) e variável alvo (y_validacao) no conjunto de validação pré-processado
# A variável alvo no conjunto de validação processado ainda está nas colunas originais do df_validacao_processed
# Precisamos extrair a coluna alvo original e mapeá-la para os rótulos numéricos usando class_mapping

# Extrair a coluna alvo original do df_validacao_processed antes do OHE (Natureza do Evento)
y_validacao_original = df_validacao_processed[target_column] # target_column foi definido anteriormente

# Mapear os rótulos da variável alvo de validação para os rótulos numéricos usando o class_mapping
# Certifique-se de que o class_mapping inclui todas as classes presentes no conjunto de validação
# Se houver classes no conjunto de validação que não estavam no conjunto de trabalho, elas resultarão em NaN
# Vamos verificar e, se houver NaNs, podemos lidar com eles (por exemplo, remover as linhas correspondentes ou atribuir uma nova categoria)
y_validacao_numeric = y_validacao_original.map(class_mapping)

# Verificar se há valores NaN após o mapeamento (indicando classes no validacao não vistas no trabalho)
nan_in_y_validacao = y_validacao_numeric.isnull().sum()
if nan_in_y_validacao > 0:
    print(f"\nAviso: {nan_in_y_validacao} valores NaN encontrados em y_validacao_numeric. Isso indica classes no conjunto de validação que não estavam no conjunto de trabalho.")
    # Remover as linhas correspondentes de X_validacao_aligned e y_validacao_numeric
    indices_to_remove_validacao = y_validacao_numeric[y_validacao_numeric.isnull()].index
    df_validacao_aligned = df_validacao_aligned.drop(index=indices_to_remove_validacao)
    y_validacao_numeric = y_validacao_numeric.drop(index=indices_to_remove_validacao)
    print(f"Removidas {len(indices_to_remove_validacao)} linhas do conjunto de validação devido a classes não vistas no treinamento.")


# X_validacao é o DataFrame de validação alinhado
X_validacao = df_validacao_aligned
# y_validacao é a série de rótulos numéricos da variável alvo
y_validacao = y_validacao_numeric.astype(int) # Converter para int

# 11. Imprimir formas e exibir primeiras linhas
print(f"\nFormato de X_validacao (features do conjunto de validação): {X_validacao.shape}")
print(f"Formato de y_validacao (variável alvo do conjunto de validação - numérica): {y_validacao.shape}")

print("\nPrimeiras linhas de X_validacao:")
display(X_validacao.head())

print("\nPrimeiras linhas de y_validacao:")
display(y_validacao.head())


DataFrame de validação alinhado com as colunas de features do conjunto de trabalho.
Tamanho do DataFrame de validação alinhado: (1520, 3774)

Aviso: 1520 valores NaN encontrados em y_validacao_numeric. Isso indica classes no conjunto de validação que não estavam no conjunto de trabalho.
Removidas 1520 linhas do conjunto de validação devido a classes não vistas no treinamento.

Formato de X_validacao (features do conjunto de validação): (0, 3774)
Formato de y_validacao (variável alvo do conjunto de validação - numérica): (0,)

Primeiras linhas de X_validacao:


,É_Feriado,Operação_Exportação,Operação_Importação,Operação_Retorno,Operação_TSF,CD Origem_BASE BRIX SANTAREM,CD Origem_BASE BRIX/POA MACAPA,CD Origem_BRIX MAO,CD Origem_Base Sequoia,CD Origem_CD 116,...,Mês_Nome_July,Mês_Nome_June,Mês_Nome_March,Mês_Nome_May,Mês_Nome_November,Mês_Nome_October,Mês_Nome_September,Periodo_Dia_Manhã,Periodo_Dia_Noite,Periodo_Dia_Tarde



Primeiras linhas de y_validacao:


,Natureza do Evento


**Reasoning**:
The validation set became empty after removing rows with classes not present in the training set. This is likely because the one-hot encoding on the validation set resulted in many new columns (new classes/categories) that were not in the training set, and then the alignment process and the subsequent removal of rows with target classes not in the training set removed all rows. To fix this, I need to re-run the preprocessing steps for the validation set, but this time ensure that the one-hot encoding is aligned with the *entire* set of columns from the one-hot encoded training set *before* separating features and target. This way, any new categories in the validation set will be added as columns (with 0 values in the training set), and the alignment will work correctly. Then, I can handle the target variable mapping.



In [ ]:
# Re-run preprocessing for validation set with improved alignment strategy

# Create copies of the original df_validacao
df_validacao_processed = df_validacao.copy()

# Convert 'Data' column to datetime
df_validacao_processed['Data'] = pd.to_datetime(df_validacao_processed['Data'], format='%d/%m/%Y')

# Extract date and time features, including 'É_Feriado'
df_validacao_processed['Dia_Semana'] = df_validacao_processed['Data'].dt.day_name()
df_validacao_processed['Mês_Nome'] = df_validacao_processed['Data'].dt.month_name()
df_validacao_processed['ANO'] = df_validacao_processed['Data'].dt.year
df_validacao_processed['Periodo_Dia'] = df_validacao_processed['Hora'].apply(periodo_do_dia)

# Ensure br_holidays is available
if 'br_holidays' not in locals():
    br_holidays = holidays.Brazil(years=df_validacao_processed['ANO'].unique())

df_validacao_processed['É_Feriado'] = df_validacao_processed['Data'].apply(is_holiday)

print("Atributos de data e hora, incluindo 'É_Feriado', extraídos for validation set.")

# Remove rows with nulls in 'CD Origem', 'Destino', 'UF DESTINO'
df_validacao_processed = df_validacao_processed.dropna(subset=['CD Origem', 'Destino', 'UF DESTINO'])
print(f"Validation set size after removing location nulls: {len(df_validacao_processed)} rows")

# Fill null values in 'Descrição da Ocorrência'
df_validacao_processed['Descrição da Ocorrência'] = df_validacao_processed['Descrição da Ocorrência'].fillna('Não Informado')
print(f"Null values in 'Descrição da Ocorrência' after filling in validation set: {df_validacao_processed['Descrição da Ocorrência'].isnull().sum()}")

# Remove the same columns removed from the training set
# Ensure 'colunas_para_remover' is defined
if 'colunas_para_remover' not in locals():
    colunas_para_remover = [
        'SM nº', 'Transportadora', 'Veículo', 'Local da Ocorrência', 'UF',
        'Semana', 'Mês', 'ANO', 'TRANSP', 'Condutor', 'Data', 'Hora'
    ]
    colunas_cd_origem_validacao = [col for col in df_validacao_processed.columns if 'CD ORIGEM' in col.upper()]
    if len(colunas_cd_origem_validacao) > 1:
        colunas_para_remover.extend(colunas_cd_origem_validacao[1:])

if 'Solicitante do Acionamento' in df_validacao_processed.columns:
     colunas_para_remover.append('Solicitante do Acionamento')

df_validacao_processed = df_validacao_processed.drop(columns=colunas_para_remover, errors='ignore')

print(f"\nRemaining columns in validation set after removal: {df_validacao_processed.columns.tolist()}")
print(f"Validation set size after column removal: {df_validacao_processed.shape}")

# Identify remaining categorical columns for One-Hot Encoding
categorical_cols_validacao = df_validacao_processed.select_dtypes(include='object').columns
print(f"\nCategorical columns identified for One-Hot Encoding in validation set: {categorical_cols_validacao.tolist()}")

# Apply One-Hot Encoding to the validation set
df_validacao_encoded = pd.get_dummies(df_validacao_processed, columns=categorical_cols_validacao, drop_first=True)

# Convert 'É_Feriado' to int
df_validacao_encoded['É_Feriado'] = df_validacao_encoded['É_Feriado'].astype(int)

print("\nValidation DataFrame after One-Hot Encoding (pre-alignment):")
display(df_validacao_encoded.head())
print(f"\nSize of validation DataFrame after One-Hot Encoding (pre-alignment): {df_validacao_encoded.shape}")
print("\nData types of validation set after One-Hot Encoding (pre-alignment):")
print(df_validacao_encoded.dtypes)

# 9. Align validation set columns with the training set columns (including target columns)
# Use the columns from the one-hot encoded training set (df_trabalho_encoded)
# This ensures that all possible categories from both train and validation are present.
# We will drop the target columns from X_validacao later.
all_encoded_cols_trabalho = df_trabalho_encoded.columns

# Reindex the encoded validation DataFrame to have the same columns as the encoded training set
# Add missing columns with value 0
df_validacao_aligned = df_validacao_encoded.reindex(columns=all_encoded_cols_trabalho, fill_value=0)

print("\nValidation DataFrame aligned with all columns from the encoded training set.")
print(f"Size of aligned validation DataFrame: {df_validacao_aligned.shape}")

# 10. Separate features (X_validacao) and target variable (y_validacao) in the processed validation set

# Identify the target columns using the target_column name
target_encoded_columns = [col for col in all_encoded_cols_trabalho if col.startswith(f'{target_column}_')]

# X_validacao are all columns except the target encoded columns
X_validacao = df_validacao_aligned.drop(columns=target_encoded_columns)

# y_validacao are the target encoded columns. We need to map them to numerical labels.
y_validacao_encoded_df = df_validacao_aligned[target_encoded_columns]

# Map the one-hot encoded target columns back to numerical labels using class_mapping
# First, find the column name with the True value for each row
y_validacao_labels = y_validacao_encoded_df.idxmax(axis=1)

# Then, map these column names to the numerical labels using the class_mapping
# Handle cases where idxmax might return a column with all False (if a row somehow ended up with no target class)
y_validacao_numeric = y_validacao_labels.map(class_mapping).fillna(-1) # Use -1 for unmapped

# Remove rows where the target mapping resulted in -1 (classes not in training set)
if (y_validacao_numeric == -1).sum() > 0:
    print(f"\nWarning: {(y_validacao_numeric == -1).sum()} rows in validation set have target classes not seen in training.")
    indices_to_remove_validacao_target = y_validacao_numeric[y_validacao_numeric == -1].index
    X_validacao = X_validacao.drop(index=indices_to_remove_validacao_target)
    y_validacao_numeric = y_validacao_numeric.drop(index=indices_to_remove_validacao_target)
    print(f"Removed {len(indices_to_remove_validacao_target)} rows from validation set due to unseen target classes.")

y_validacao = y_validacao_numeric.astype(int)

# 11. Print shapes and display first rows
print(f"\nShape of X_validacao (validation set features): {X_validacao.shape}")
print(f"Shape of y_validacao (validation set target - numerical): {y_validacao.shape}")

print("\nFirst rows of X_validacao:")
display(X_validacao.head())

print("\nFirst rows of y_validacao:")
display(y_validacao.head())

Atributos de data e hora, incluindo 'É_Feriado', extraídos for validation set.
Validation set size after removing location nulls: 1520 rows
Null values in 'Descrição da Ocorrência' after filling in validation set: 0

Remaining columns in validation set after removal: ['Operação', 'CD Origem', 'Destino', 'Natureza do Evento', 'Causa Raiz', 'Processo Macro', 'Descrição da Ocorrência', 'Pronta Resposta', 'UF DESTINO', 'Dia_Semana', 'Mês_Nome', 'Periodo_Dia', 'É_Feriado']
Validation set size after column removal: (1520, 13)

Categorical columns identified for One-Hot Encoding in validation set: ['Operação', 'CD Origem', 'Destino', 'Natureza do Evento', 'Causa Raiz', 'Processo Macro', 'Descrição da Ocorrência', 'Pronta Resposta', 'UF DESTINO', 'Dia_Semana', 'Mês_Nome', 'Periodo_Dia']

Validation DataFrame after One-Hot Encoding (pre-alignment):


,É_Feriado,Operação_Exportação,Operação_Retorno,Operação_TSF,CD Origem_BASE BRIX/POA MACAPA,CD Origem_CD 116,CD Origem_CD 116 DSL Ananindeu,CD Origem_CD 116 DSL FORTALEZA,CD Origem_CD 116 DSL J. PESSOA,CD Origem_CD 116 DSL Posto 13,...,Mês_Nome_July,Mês_Nome_June,Mês_Nome_March,Mês_Nome_May,Mês_Nome_November,Mês_Nome_October,Mês_Nome_September,Periodo_Dia_Manhã,Periodo_Dia_Noite,Periodo_Dia_Tarde
1321,1,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,True,False
3520,0,False,False,False,False,True,False,False,False,False,...,False,False,True,False,False,False,False,True,False,False
3176,0,False,False,False,False,True,False,False,False,False,...,True,False,False,False,False,False,False,False,True,False
132,0,False,False,False,False,False,False,False,False,False,...,True,False,False,False,False,False,False,False,False,True
1430,0,False,False,False,False,True,False,False,False,False,...,False,False,True,False,False,False,False,False,False,True



Size of validation DataFrame after One-Hot Encoding (pre-alignment): (1520, 1258)

Data types of validation set after One-Hot Encoding (pre-alignment):
É_Feriado                         int64
Operação_Exportação                bool
Operação_Retorno                   bool
Operação_TSF                       bool
CD Origem_BASE BRIX/POA MACAPA     bool
                                  ...  
Mês_Nome_October                   bool
Mês_Nome_September                 bool
Periodo_Dia_Manhã                  bool
Periodo_Dia_Noite                  bool
Periodo_Dia_Tarde                  bool
Length: 1258, dtype: object

Validation DataFrame aligned with all columns from the encoded training set.
Size of aligned validation DataFrame: (1520, 3828)

Shape of X_validacao (validation set features): (1520, 3774)
Shape of y_validacao (validation set target - numerical): (1520,)

First rows of X_validacao:


,É_Feriado,Operação_Exportação,Operação_Importação,Operação_Retorno,Operação_TSF,CD Origem_BASE BRIX SANTAREM,CD Origem_BASE BRIX/POA MACAPA,CD Origem_BRIX MAO,CD Origem_Base Sequoia,CD Origem_CD 116,...,Mês_Nome_July,Mês_Nome_June,Mês_Nome_March,Mês_Nome_May,Mês_Nome_November,Mês_Nome_October,Mês_Nome_September,Periodo_Dia_Manhã,Periodo_Dia_Noite,Periodo_Dia_Tarde
1321,1,False,0,False,False,0,False,0,0,False,...,False,False,False,False,False,False,False,False,True,False
3520,0,False,0,False,False,0,False,0,0,True,...,False,False,True,False,False,False,False,True,False,False
3176,0,False,0,False,False,0,False,0,0,True,...,True,False,False,False,False,False,False,False,True,False
132,0,False,0,False,False,0,False,0,0,False,...,True,False,False,False,False,False,False,False,False,True
1430,0,False,0,False,False,0,False,0,0,True,...,False,False,True,False,False,False,False,False,False,True



First rows of y_validacao:


,0
1321,20
3520,13
3176,29
132,47
1430,30


## Treinamento dos modelos

### Subtask:
Treinar os modelos selecionados utilizando o conjunto de trabalho balanceado (`X_res`, `y_res_numeric`).
